In [ ]:
%load_ext autoreload
%autoreload 2

# Inspect Dataset — Snapshot Kgalagadi

Visual inspection of the curated Kgalagadi camera trap subset.  
Covers: class distribution, resolution spread, sample grids, ImageFolder loading, DataLoader batch.

## Imports

In [ ]:
import csv
import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import pyrootutils
import seaborn as sns
import torch
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid

## Parameters

In [ ]:
ROOT = pyrootutils.setup_root(
    search_from=".",
    indicator="pyproject.toml",
    project_root_env_var=True,
    dotenv=True,
    pythonpath=True,
    cwd=True,
)

In [ ]:
DATA_DIR = ROOT / "data/kgalagadi"
SPLITS = ["train", "val", "test"]
SEED = 0

assert DATA_DIR.exists(), f"Not found: {DATA_DIR.resolve()}"
print(f"Dataset path: {DATA_DIR.resolve()}")

## Section 1 — Class Distribution

Count images per class and split from `metadata.csv`, then plot a bar chart.

In [ ]:
with (DATA_DIR / "metadata.csv").open() as f:
    rows = list(csv.DictReader(f))

print(f"Total images: {len(rows)}")
print(f"Columns: {list(rows[0].keys())}\n")

# Per-split, per-class counts
counts = defaultdict(Counter)  # counts[split][label]
for r in rows:
    counts[r["split"]][r["label"]] += 1

all_labels = sorted({r["label"] for r in rows})
col_w = 8
print(
    f"{'label':<25}" + "".join(f"{s:>{col_w}}" for s in SPLITS) + f"{'total':>{col_w}}"
)
print("-" * (25 + col_w * (len(SPLITS) + 1)))
for label in all_labels:
    cs = [counts[s][label] for s in SPLITS]
    print(f"{label:<25}" + "".join(f"{c:>{col_w}}" for c in cs) + f"{sum(cs):>{col_w}}")
totals = [sum(counts[s].values()) for s in SPLITS]
print(
    f"{'TOTAL':<25}"
    + "".join(f"{t:>{col_w}}" for t in totals)
    + f"{sum(totals):>{col_w}}"
)

In [ ]:
train_counts = [counts["train"][label] for label in all_labels]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(all_labels, train_counts, color="steelblue", edgecolor="white")
_ = ax.bar_label(bars, fmt="%d", padding=3)
_ = ax.set_title("Kgalagadi — train images per class", fontsize=13)
_ = ax.set_ylabel("Image count")
_ = ax.set_xlabel("Class")
_ = plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

max_c, min_c = max(train_counts), min(train_counts)
print(
    f"Max/min ratio (train): {max_c}/{min_c} = {max_c / min_c:.1f}x  — heavy imbalance expected (empty dominates)"
)

## Section 2 — Resolution Distribution

All Kgalagadi images were resized to max 1024 px on the longest side. Confirm the spread.

In [ ]:
widths = [int(r["width"]) for r in rows]
heights = [int(r["height"]) for r in rows]

print(f"Width  — min: {min(widths)}, max: {max(widths)}, unique: {len(set(widths))}")
print(f"Height — min: {min(heights)}, max: {max(heights)}, unique: {len(set(heights))}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
_ = sns.histplot(widths, bins=20, color="steelblue", ax=axes[0])
_ = axes[0].set_title("Width distribution")
_ = axes[0].set_xlabel("Pixels")
_ = sns.histplot(heights, bins=20, color="coral", ax=axes[1])
_ = axes[1].set_title("Height distribution")
_ = axes[1].set_xlabel("Pixels")
plt.suptitle("Kgalagadi — image resolution", fontsize=12)
plt.tight_layout()
plt.show()

## Section 3 — Sample Grid per Class

4 × 4 random samples from the training split for each class.  
Check for: wrong labels, pure-IR night images, extreme exposure.

In [ ]:
random.seed(SEED)

# Build index: label -> list of train image paths
train_index = defaultdict(list)
for split_dir in (DATA_DIR / "train",):
    for label_dir in sorted(split_dir.iterdir()):
        if label_dir.is_dir():
            train_index[label_dir.name] = sorted(label_dir.iterdir())

N_COLS, N_ROWS = 4, 4
for label in all_labels:
    paths = train_index.get(label, [])
    sample = random.sample(paths, min(N_COLS * N_ROWS, len(paths)))
    n = len(sample)
    n_cols, n_rows = N_COLS, (n + N_COLS - 1) // N_COLS

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 2.5))
    fig.suptitle(f"class: {label}  (n_train={len(paths)})", fontsize=12, y=1.01)
    axes_flat = axes.flatten() if n > 1 else [axes]

    for ax, p in zip(axes_flat, sample, strict=False):
        with Image.open(p) as im:
            _ = ax.imshow(im)
        ax.axis("off")
    for ax in axes_flat[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Section 4 — ImageFolder Loading

Verify that torchvision's `ImageFolder` reads the dataset correctly.

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_ds = ImageFolder(DATA_DIR / "train", transform=transform)
val_ds = ImageFolder(DATA_DIR / "val", transform=transform)
test_ds = ImageFolder(DATA_DIR / "test", transform=transform)

print(f"Classes:    {train_ds.classes}")
print(f"Train size: {len(train_ds)}")
print(f"Val size:   {len(val_ds)}")
print(f"Test size:  {len(test_ds)}")

img, label = train_ds[0]
print(f"\nSample tensor — shape: {img.shape}, dtype: {img.dtype}")
print(f"Value range:  [{img.min():.2f}, {img.max():.2f}]  (normalised, not [0,1])")

## Section 5 — DataLoader Batch

Load one batch and display it as a grid (after un-normalising for visualisation).

In [ ]:
loader = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED),
)

imgs, labels = next(iter(loader))
print(f"Batch shape:  {imgs.shape}")
print(f"Labels:       {[train_ds.classes[label] for label in labels.tolist()]}")

In [ ]:
# Un-normalise for display
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
imgs_disp = (imgs * std + mean).clamp(0, 1)

grid = make_grid(imgs_disp, nrow=8, padding=4)
fig, ax = plt.subplots(figsize=(18, 5))
_ = ax.imshow(grid.permute(1, 2, 0).numpy())
_ = ax.set_title(
    "Kgalagadi — one training batch (224x224, normalised then un-normalised)",
    fontsize=12,
)
ax.axis("off")
plt.tight_layout()
plt.show()